# Theme extraction

IMPORTANT: PLEASE 

## 0. A small introduction to theme extraction

To extract themes from both the contest images (via their textual descriptions) and the user-submitted captions in a scalable and linguistically meaningful way, we rely on **sentence-level embeddings** rather than traditional bag-of-words topic models. Our pipeline is presented like so:

- Sentence-BERT embeddings
- Unsupervised clustering
- Theme assignment (for both images and captions)

Each description or caption is converted into a dense vector representation using a **Sentence-BERT** model, which captures contextual meaning, humor, and implicit relationships far better than word-frequency-based methods. Our approach is particularly well suited for caption data, which is typically short, creative, and semantically rich.

We initially considered classical topic modeling approaches like **Latent Dirichlet Allocation (LDA)**, but LDA’s bag-of-words assumptions and poor performance on short, diverse, humor-oriented texts make it unsuitable for this dataset. We also evaluated **BERTopic**, which combines embeddings with topic modeling, but its reliance on UMAP makes it difficult to scale efficiently to the size of our dataset.

Once all texts have been embedded into a shared semantic space, we apply **unsupervised clustering** methods (e.g., MiniBatchKMeans, HDBSCAN) to group semantically similar texts into coherent themes that reflect recurring patterns across contests. By clustering all descriptions and captions globally, rather than per contest, we create a unified thematic space that remains consistent throughout the dataset—an essential property for cross-contest comparisons and downstream analyses.

This embedding-based clustering pipeline enables us to assign each caption and each contest image to a specific **theme_id**, producing a structured and interpretable representation of the humor of the captions and visual concepts present in the dataset. These theme assignments provide the foundation for following analyses such as exploring funniness-theme correlations, measuring image–caption alignment, and visualizing global humor dynamics.

In [18]:
import json
from pathlib import Path

from tqdm.auto import tqdm

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans

In [16]:
# General variables: paths
DATA_ROOT = Path("../../data/newyorker_caption_contest")
CONTESTS_JSON = DATA_ROOT / "contests.json"
DATA_DIR = DATA_ROOT / "data"

# Sentence-BERT model
BERT_MODEL = "sentence-transformers/all-mpnet-base-v2"

## 1. Loading the data

We start by loading our data in a unified way in a custom pandas dataframe.

In [ ]:
def load_contests(contests_json_path: Path):
    with open(contests_json_path, "r", encoding="utf-8") as f:
        contests = json.load(f)
    return contests

def build_docs_dataframe():
    contests = load_contests(CONTESTS_JSON)
    docs = []

    # tqdm over contests
    for contest in tqdm(contests, desc="Step 1: data loading/building"):
        cid = contest["contest_id"]

        raw_meta = contest.get("metadata")
        meta = raw_meta or {}   # normalize None -> {}

        # ---- Build one aggregated "image document" per contest ----
        parts = []

        for key in ["image_descriptions", "image_uncanny_descriptions",
                    "image_locations", "questions"]:
            values = meta.get(key) or []
            parts.extend(values)

        entities = meta.get("entities") or []
        for url in entities:
            last = str(url).rstrip("/").split("/")[-1]
            parts.append(last.replace("_", " "))

        image_text = " ".join(map(str, parts)).strip()

        if image_text:
            docs.append(
                {
                    "doc_id": f"image_{cid}",
                    "contest_id": cid,
                    "kind": "image",
                    "text": image_text,
                }
            )

        # ---- Add caption documents for this contest ----
        csv_path = DATA_DIR / f"{cid}.csv"
        if not csv_path.exists():
            print(f"Warning: CSV for contest {cid} not found at {csv_path}")
            continue

        captions_df = pd.read_csv(csv_path)

        # optional inner tqdm if you want per-caption progress too
        for idx, row in captions_df.iterrows():
            cap_text = str(row.get("caption", "")).strip()
            if not cap_text:
                continue

            docs.append(
                {
                    "doc_id": f"caption_{cid}_{idx}",
                    "contest_id": cid,
                    "kind": "caption",
                    "text": cap_text,
                    "rank": row.get("rank", None),
                    "mean": row.get("mean", None),
                    "precision": row.get("precision", None),
                    "votes": row.get("votes", None),
                    "not_funny": row.get("not_funny", None),
                    "somewhat_funny": row.get("somewhat_funny", None),
                    "funny": row.get("funny", None),
                }
            )

    docs_df = pd.DataFrame(docs)
    return docs_df


In [15]:
docs_df = build_docs_dataframe()

print(docs_df.head())

print("Total documents:", len(docs_df))
print(docs_df["kind"].value_counts())

Building docs from contests:   0%|          | 0/384 [00:00<?, ?it/s]

          doc_id  contest_id     kind  \
0      image_510         510    image   
1  caption_510_0         510  caption   
2  caption_510_1         510  caption   
3  caption_510_2         510  caption   
4  caption_510_3         510  caption   

                                                text  rank      mean  \
0  A man is relaxing on a city street. Others are...   NaN       NaN   
1          I'm a congressman--obstruction is my job.   0.0  1.913043   
2  I'm what they mean when they say, 'The middle ...   1.0  1.842105   
3                  Does this suit make me look flat?   2.0  1.711111   
4    When the right woman comes along, I'll know it.   3.0  1.625000   

   precision  votes  not_funny  somewhat_funny  funny  
0        NaN    NaN        NaN             NaN    NaN  
1   0.094022   69.0       24.0            27.0   18.0  
2   0.191381   19.0        8.0             6.0    5.0  
3   0.112915   45.0       21.0            16.0    8.0  
4   0.116657   32.0       15.0          

## 2. Sentence-BERT encoding

This step will take a lot of time to process.

So go grab a coffee, open a good book, or a Selling Sunset episode ... it'll definitely speed up the process. Or invent time travelling, maybe that would work better... maybe... who knows...

<!-- oooor I could tell you about some gossip I heard the other day at the cafetaria! Aaaah, now you're interested. 

""" <br>
So basically, a friend of mine broke up with her boyfriend and slept with his guy who she had the biggest crush on for YEARS. A few days later she started getting a burning in her vagina. She thought it was a yeast infection. NOPE! It was herpes.

However, the guy she slept with had never had a herpes outbreak prior to the current one so his herpes was new too. Where did guy A get the herpes from? He slept with a woman who had a recent herpes outbreak. This woman also slept with guy A's FATHER and is now pregnant!

But where did the herpes originally come from? From guy A's father.

So to sum up this woman is now pregnant, they don't know if the childs father is father or son, and now everyone has herpes. 
<br>""" -->

In [17]:
model = SentenceTransformer(BERT_MODEL)

def compute_embeddings(texts, batch_size=64):
    # returns a numpy array of shape (N, D)
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # helpful for cosine similarity
    )
    return embeddings

embeddings = compute_embeddings(docs_df["text"].tolist())
print("Embeddings shape:", embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/35827 [00:00<?, ?it/s]

Embeddings shape: (2292901, 768)


## 3. Global clustering (across all captions + images)

We’ll do global clustering across all documents (captions + images), so they share the same theme space.

For scalability, we will use MiniBatchKMeans. To start we will choose a number of clusters of ```n_clusters=80``` and fine tune it afterwards.

In [ ]:
N_CLUSTERS = 80 

kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    batch_size=2048,
    max_iter=100,
    random_state=42,
)
kmeans.fit(embeddings)

theme_labels = kmeans.predict(embeddings)
docs_df["theme_id"] = theme_labels


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## 4. Add results to our dataframe

In [ ]:
# full docs with themes
docs_with_themes = docs_df.copy()

# captions only
caption_themes = docs_with_themes[docs_with_themes["kind"] == "caption"].copy()

# images only
image_themes = docs_with_themes[docs_with_themes["kind"] == "image"].copy()

print(caption_themes.head())
print(image_themes.head())

          doc_id  contest_id     kind  \
1  caption_510_0         510  caption   
2  caption_510_1         510  caption   
3  caption_510_2         510  caption   
4  caption_510_3         510  caption   
5  caption_510_4         510  caption   

                                                text  rank      mean  \
1          I'm a congressman--obstruction is my job.   0.0  1.913043   
2  I'm what they mean when they say, 'The middle ...   1.0  1.842105   
3                  Does this suit make me look flat?   2.0  1.711111   
4    When the right woman comes along, I'll know it.   3.0  1.625000   
5  I used to lie in the gutter, but then I quit d...   4.0  1.617647   

   precision  votes  not_funny  somewhat_funny  funny  theme_id  
1   0.094022   69.0       24.0            27.0   18.0        75  
2   0.191381   19.0        8.0             6.0    5.0        37  
3   0.112915   45.0       21.0            16.0    8.0        58  
4   0.116657   32.0       15.0            14.0    3.0   

## 5. Theme definitions: TF–IDF-based labels per theme_id

Now that we have groups of themes, we want to find what describeds each of them accordingly by finding keywords to described each one of them.

Idea:
- Fit a single TF–IDF over all texts.
- For each theme (cluster), compute the average TF–IDF vector over its documents.
- Take the top N terms → that’s your label candidate.

### 5.1. Fit TF–IDF on all docs

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

# fine-tune 
MAX_FEATURES = 20000
MIN_DF = 5   # ignore words that appear in <5 documents
NGRAM_RANGE = (1, 2)  # unigrams + bigrams should work well for themes

vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    min_df=MIN_DF,
    ngram_range=NGRAM_RANGE,
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(docs_with_themes["text"])
feature_names = np.array(vectorizer.get_feature_names_out())

print(tfidf_matrix.shape)  # (n_docs, n_features)


(2292901, 20000)


### 5.2. Get top TF–IDF terms per cluster

We’ll compute, for each theme_id:
- the mean TF–IDF vector across its docs
- the top-k features with highest mean score

In [ ]:
from collections import defaultdict

def get_top_terms_per_theme(docs_df, tfidf_matrix, feature_names, top_k=8):
    theme_to_terms = {}
    theme_ids = sorted(docs_df["theme_id"].unique())

    for theme_id in theme_ids:
        mask = (docs_df["theme_id"] == theme_id).values
        if mask.sum() == 0:
            continue

        # slice TF-IDF rows for this theme
        theme_tfidf = tfidf_matrix[mask]

        # compute mean tf-idf across docs in this theme (1 x n_features)
        mean_tfidf = theme_tfidf.mean(axis=0)  # still a sparse matrix
        mean_tfidf = np.asarray(mean_tfidf).ravel()

        # get top_k indices
        if np.all(mean_tfidf == 0):
            # cluster with weird texts, skip
            theme_to_terms[theme_id] = []
            continue

        top_idx = mean_tfidf.argsort()[::-1][:top_k]
        top_terms = feature_names[top_idx].tolist()
        theme_to_terms[theme_id] = top_terms

    return theme_to_terms

theme_to_terms = get_top_terms_per_theme(
    docs_with_themes, tfidf_matrix, feature_names, top_k=8
)

# display the 5 first themes
for tid in list(theme_to_terms.keys())[:5]:
    print(tid, "->", theme_to_terms[tid])


0 -> ['taste', 'like', 'recipe', 'smell', 'fresh', 'just', 'tastes', 'good']
1 -> ['ll', 'think', 'just', 'going', 'let', 'don', 'like', 'got']
2 -> ['clown', 'clowns', 'circus', 'funny', 'clowning', 'cop', 'problem', 'don']
3 -> ['room', 'space', 'floor', 'place', 'wall', 'table', 'ikea', 'cave']
4 -> ['water', 'beach', 'whale', 'swim', 'sea', 'island', 'boat', 'pool']


### 5.3. Turn top terms into a short textual label

Now we compress the list of terms into a short label string.

In [24]:
def auto_label_from_terms(terms, max_words=3):
    if not terms:
        return "miscellaneous"
    # prefer bigrams if possible, then unigrams
    bigrams = [t for t in terms if " " in t]
    unigrams = [t for t in terms if " " not in t]

    label_terms = bigrams[:max_words]
    if len(label_terms) < max_words:
        label_terms += unigrams[: max_words - len(label_terms)]

    # simple title-case label
    label = ", ".join(label_terms)
    return label.title()

In [ ]:
theme_to_label = {
    tid: auto_label_from_terms(terms)
    for tid, terms in theme_to_terms.items()
}

# display the 10 first themes
for tid in list(theme_to_label.keys())[:10]:
    print(tid, "=>", theme_to_label[tid])

docs_with_themes["theme_label"] = docs_with_themes["theme_id"].map(theme_to_label)
docs_with_themes.head()


0 => Taste, Like, Recipe
1 => Ll, Think, Just
2 => Clown, Clowns, Circus
3 => Room, Space, Floor
4 => Water, Beach, Whale
5 => Amazon, Ordered, Sale
6 => Movie, Like, Just
7 => Know, Yes, Ve
8 => Insurance, Pay, Taxes
9 => Relationship, Just, Think


,doc_id,contest_id,kind,text,rank,mean,precision,votes,not_funny,somewhat_funny,funny,theme_id,theme_label
0,image_510,510,image,A man is relaxing on a city street. Others are...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37,"Think, Don, Just"
1,caption_510_0,510,caption,I'm a congressman--obstruction is my job.,0.0,1.913043,0.094022,69.0,24.0,27.0,18.0,75,"Trump, President, Election"
2,caption_510_1,510,caption,"I'm what they mean when they say, 'The middle ...",1.0,1.842105,0.191381,19.0,8.0,6.0,5.0,37,"Think, Don, Just"
3,caption_510_2,510,caption,Does this suit make me look flat?,2.0,1.711111,0.112915,45.0,21.0,16.0,8.0,58,"Look Like, Look, Like"
4,caption_510_3,510,caption,"When the right woman comes along, I'll know it.",3.0,1.625000,0.116657,32.0,15.0,14.0,3.0,68,"Date, Dating, Just"


In [29]:
# display all unique themes_label
docs_with_themes["theme_label"].unique()

array(['Think, Don, Just', 'Trump, President, Election',
       'Look Like, Look, Like', 'Date, Dating, Just', 'Don, Just, Way',
       'Therapy, Therapist, Help', 'Elevator, Way, Stairs',
       'Size, Feet, Yoga', 'Don, Ll, Time', 'Feel Like, Feel, Like',
       'Don, Just, Think', 'Guy, New, Just', 'Know, Yes, Ve',
       'Dinner, Lunch, Ll', 'Room, Space, Floor', 'Ll, Want, Like',
       'Art, Artist, Painting', 'Time, Day, Today',
       'House, Apartment, Neighborhood', 'Music, Song, Play',
       'Lawyer, Witness, Case', 'App, Cloud, Just', 'View, Look, Eyes',
       'Job, Work, Resume', 'Movie, Like, Just', 'Pants, Shoes, Wear',
       'Doctor, Doc, Patient', 'Watson, Think, Word',
       'Dress, Wear, Wearing', 'Office, Boss, Job', 'Moon, Star, Stars',
       'Going, Time, Think', 'Bird, Birds, Pigeon',
       'Earth, Space, Planet', 'Tip, Ll, Pay', 'Dog, Dogs, Pet',
       'Hear, Say, Said', 'Trophy Wife, Wife, Divorce',
       'Gluten Free, Diet, Gluten', 'Covid, Pandemic, Q

## 6. Saving results to csv

In [30]:
# save results to CSV
docs_with_themes.to_csv("../../data/newyorker_caption_contest/themes_extraction_csv/docs_with_themes.csv", index=False)